# Variational Circuits

This notebook covers symbolic parameters, parameter binding, sweeps, and
OpenQASM round-trips.

In [1]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/circuit/param"
	"github.com/splch/goqu/qasm/emitter"
	"github.com/splch/goqu/qasm/parser"
	"github.com/splch/goqu/sweep"
)

In [2]:
var theta = param.New("theta")

func buildVariational() *ir.Circuit {
	c, _ := builder.New("variational", 2).
		SymRY(theta.Expr(), 0).
		CNOT(0, 1).
		Build()
	return c
}

func bindVariational(val float64) *ir.Circuit {
	bound, _ := ir.Bind(buildVariational(), map[string]float64{"theta": val})
	return bound
}

## Symbolic Parameters

Create a parameter with `param.New()` and use it in symbolic rotation gates.

In [3]:
%%
c := buildVariational()
fmt.Println("Free parameters:", ir.FreeParameters(c))
gonbui.DisplaySvg(draw.SVG(c))

Free parameters: [theta]


q0 
 
 q1 
 
 RY

## Binding Parameters

`ir.Bind()` substitutes symbolic parameters with concrete values, producing a
new circuit.

In [4]:
%%
bound := bindVariational(math.Pi / 4)
fmt.Println("Free parameters after bind:", ir.FreeParameters(bound))
gonbui.DisplaySvg(draw.SVG(bound))

Free parameters after bind: []


q0 
 
 q1 
 
 RY(pi/4)

## Parameter Sweeps

Sweep a parameter across a range and collect measurement counts for each point.

In [5]:
%%
cSweep, _ := builder.New("sweep", 2).
	SymRY(theta.Expr(), 0).
	CNOT(0, 1).
	MeasureAll().
	Build()

sw := sweep.Linspace{Key: "theta", Start: 0, Stop: math.Pi, Count: 5}
results, _ := sweep.RunSim(context.Background(), cSweep, 1000, sw)

html := "<table><tr><th>theta</th><th>|00&gt;</th><th>|11&gt;</th></tr>\n"
for _, r := range results {
	html += fmt.Sprintf("<tr><td>%.2f</td><td>%d</td><td>%d</td></tr>\n",
		r.Bindings["theta"], r.Counts["00"], r.Counts["11"])
}
html += "</table>"
gonbui.DisplayHTML(html)

theta,|00>,|11>
0.00,1000,0
0.79,863,137
1.57,502,498
2.36,150,850
3.14,0,1000


## OpenQASM Round-Trip

Emit a bound circuit as OpenQASM 3.0, then parse it back.

In [6]:
%%
qasm, _ := emitter.EmitString(bindVariational(math.Pi / 4))
fmt.Println(qasm)

OPENQASM 3.0;
include "stdgates.inc";
qubit[2] q;

ry(pi/4) q[0];
cx q[0], q[1];



In [7]:
%%
qasm, _ := emitter.EmitString(bindVariational(math.Pi / 4))
roundTrip, _ := parser.ParseString(qasm)
fmt.Printf("Round-trip circuit: %s (%d qubits, depth %d)\n",
	roundTrip.Name(), roundTrip.NumQubits(), roundTrip.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(roundTrip))

Round-trip circuit:  (2 qubits, depth 2)


q0 
 
 q1 
 
 RY(pi/4)